# [Lab0] 환경 설정 및 초기화

이 노트북에서는 SageMaker 엔드투엔드 머신러닝 워크플로우를 위한 환경을 설정합니다.

## 주요 설정 내용
- AWS 환경 및 권한 설정
- MLflow 추적 서버 연결
- 데이터 준비 및 S3 업로드
- 실험 환경 초기화

## 1. 필수 라이브러리 및 AWS 환경 설정

In [ ]:
# 이 워크샵은 classic SageMaker Python SDK (v2)를 사용합니다.
# 커널 이미지에는 모듈형/v3 패키지만 있어 sagemaker.sklearn 등이 없으므로 classic v2를 설치합니다.
# (FrameworkProcessor, Estimator, Pipelines 등에 필요)
# setuptools<81 핀: setuptools 81+ 는 pkg_resources를 제거해 mlflow 2.13.2 import가 깨지므로 고정
# 이미 설치돼 있으면 건너뜁니다 → Run All 시 무한 재시작 방지.
import importlib.util

if importlib.util.find_spec("sagemaker.sklearn") is None:
    %pip install -q "setuptools<81" "sagemaker>=2.220,<3" mlflow==2.13.2 sagemaker-mlflow
    print("✅ 설치 완료 — 커널을 재시작합니다. 재시작 후 다시 [Run All] 하세요.")
    import IPython
    IPython.Applicatiㅁon.instance().kernel.do_shutdown(True)
else:
    print("✅ classic SageMaker SDK 사용 가능 (설치 건너뜀)")

In [ ]:
# (설치와 커널 재시작은 위 셀에서 자동 처리됩니다.)
# 재시작이 필요했다면 위 셀이 커널을 내렸다가 올리며, 그 후 [Run All]을 다시 실행하면
# 이 셀은 그냥 통과합니다. 수동 재시작이 필요할 때만 아래 두 줄의 주석을 해제하세요.
# import IPython
# IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# 필수 라이브러리 임포트
import logging
from time import gmtime, strftime

import boto3
import pandas as pd
import mlflow

from sagemaker_studio import Project

print("✅ 라이브러리 임포트 완료")


In [ ]:
# 로깅 설정
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✅ 로깅 설정 완료")

In [ ]:
# AWS 세션 설정
boto_session = boto3.Session()
region = boto_session.region_name

print(f"✅ AWS 리전: {region}")


## 2. SageMaker AI 프로젝트 설정

SageMaker AI 프로젝트를 통해 통합된 리소스 관리를 수행합니다.

In [ ]:
# SageMaker AI 프로젝트 초기화
project = Project()

# MLflow 추적 서버 ARN 가져오기
# 주의: project.mlflow_tracking_server_arn 은 environment 이름이 정확히
#       "MLExperiments" 일 때만 동작합니다. 실제 프로젝트의 environment 이름이
#       "OnCreate MLExperiments" 처럼 다를 경우 'MLExperiments environment not found'
#       에러가 발생하므로, DataZone API로 직접(부분 일치) 조회합니다.
dz = boto3.client("datazone", region_name=region)
domain_id = project.domain_id
project_id = project.id

mlflow_arn = None
paginator = dz.get_paginator("list_environments")
for page in paginator.paginate(domainIdentifier=domain_id, projectIdentifier=project_id):
    for env in page.get("items", []):
        if "MLExperiments" in env.get("name", ""):
            env_detail = dz.get_environment(domainIdentifier=domain_id, identifier=env["id"])
            for res in env_detail.get("provisionedResources", []):
                if res.get("name") == "mlflowTrackingServerArn":
                    mlflow_arn = res.get("value")

if not mlflow_arn:
    raise RuntimeError("MLflow tracking server ARN을 찾지 못했습니다. MLExperiments environment를 확인하세요.")

mlflow_name = mlflow_arn.split("tracking-server/")[-1]

# MLflow 추적 URI 설정
mlflow.set_tracking_uri(mlflow_arn)

print(f"✅ MLflow 추적 서버 ARN: {mlflow_arn}")
print(f"✅ MLflow 서버 이름: {mlflow_name}")


In [ ]:
# S3 버킷 및 IAM 역할 설정
bucket_root = project.s3.root
role = project.iam_role

# S3 URI 파싱
s3_parts = bucket_root.replace("s3://", "").split("/")
bucket = s3_parts[0]
prefix = "/".join(s3_parts[1:] + ["sagemaker/DEMO-xgboost-dm"])

print(f"✅ S3 버킷: {bucket}")
print(f"✅ S3 프리픽스: {prefix}")
print(f"✅ IAM 역할: {role}")

In [ ]:
# 프로젝트 데이터베이스 정보 가져오기
catalog = project.connection().catalog()
project_database = catalog.databases[0].name

print(f"✅ 프로젝트 데이터베이스: {project_database}")

## 3. 실험 설정

MLflow를 사용한 실험 추적을 위한 설정을 수행합니다.

In [ ]:
# 실험 이름 생성 및 MLflow 실험 설정
experiment_suffix = strftime('%d-%H-%M-%S', gmtime())
experiment_name = f"end-to-end-experiment-{experiment_suffix}"

mlflow.set_experiment(experiment_name)

print(f"✅ 실험 이름: {experiment_name}")

## 4. 데이터 준비

은행 마케팅 데이터셋을 준비하고 기본적인 탐색을 수행합니다.

In [ ]:
import os
import zipfile

# 데이터 압축 파일이 없으면 다운로드
if not os.path.exists('bank-additional.zip'):
    print("bank-additional.zip 이 없어 다운로드합니다...")
    !wget -q https://sagemaker-sample-data-us-west-2.s3-us-west-2.amazonaws.com/autopilot/direct_marketing/bank-additional.zip

# 데이터 압축 해제
with zipfile.ZipFile('bank-additional.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

print("✅ 데이터 압축 해제 완료")

In [ ]:
# 데이터 로드 및 기본 정보 확인
data = pd.read_csv('./bank-additional/bank-additional-full.csv')

print(f"✅ 데이터 로드 완료")
print(f"   - 데이터 크기: {data.shape}")
print(f"   - 컬럼 수: {len(data.columns)}")
print(f"   - 타겟 변수 분포:")
print(data['y'].value_counts())

# 처음 5행 표시
print("\n📊 데이터 미리보기:")
display(data.head())

## 5. 변수 저장

다음 노트북에서 사용할 중요한 변수들을 저장합니다.

In [ ]:
# 중요한 변수들을 다음 노트북에서 사용하기 위해 저장
%store bucket
%store prefix
%store role
%store region
%store mlflow_arn
%store mlflow_name
%store experiment_name

print("✅ 모든 변수가 저장되었습니다.")
print("\n📋 저장된 변수 요약:")
print(f"   - S3 버킷: {bucket}")
print(f"   - S3 프리픽스: {prefix}")
print(f"   - 리전: {region}")
print(f"   - 실험 이름: {experiment_name}")
print(f"   - MLflow 서버: {mlflow_name}")

## ✅ 설정 완료

환경 설정이 완료되었습니다! 이제 다음 노트북 `1-preprocessing.ipynb`로 진행하여 데이터 전처리를 수행할 수 있습니다.

### 다음 단계
1. **데이터 전처리** (`1-preprocessing.ipynb`): SageMaker Processing을 사용한 데이터 전처리
2. **모델 훈련** (`2-training.ipynb`): SageMaker Training Job을 사용한 모델 훈련
3. **모델 평가** (`3-model-evaluation.ipynb`): 두 모델 성능 비교 및 최고 모델 선택
4. **모델 배포** (`4-test-and-deploy.ipynb`): 모델 엔드포인트 배포 및 테스트
5. **파이프라인 구축** (`5-pipelines_preprocess_train_evaluate_batch_transform.ipynb`): SageMaker Pipelines 예제